# 🔐 Data Management in Databricks — Project Showcase

![Databricks](https://img.shields.io/badge/Platform-Databricks-FF3621?logo=databricks&logoColor=white)
![Delta Lake](https://img.shields.io/badge/Storage-Delta%20Lake-00ADD8)
![Dataset](https://img.shields.io/badge/Dataset-Healthcare%20Records-blueviolet)
![Status](https://img.shields.io/badge/Status-Completed-brightgreen)

**Scenario I worked through:** acting as a data engineer at a healthcare company (**HealthInc**), responsible for keeping patient, prescription, and appointment data reliable, performant, and properly governed inside **Delta Lake** — where correctness and access control genuinely matter, since the underlying records are patient data.

**What this notebook demonstrates:** safe, ACID-compliant updates and deletes on Delta tables; the practical difference between managed and unmanaged tables; building both persistent and temporary views; and applying real access-control and PII-protection practices through Databricks' Data Explorer.

---


## Chapter 1 — Delta Lake, ACID Transactions & Table Persistence

Delta Lake underpins every table in Databricks. The core promise is **ACID compliance** (Atomicity, Consistency, Isolation, Durability): every update either fully succeeds or fully fails, which is exactly the guarantee you want when the data represents patient records. I also learned the difference between **managed tables** (Databricks owns the data's lifecycle) and **unmanaged tables** (the data lives at a custom `LOCATION` and survives even if the table definition is dropped) — a distinction with real consequences for storage cost, compliance, and cleanup.


### 1.1 — Updating a patient record with an ACID-safe transaction

A patient's phone number in the `patients` table was incorrect. Rather than editing the file directly, I ran a governed `UPDATE` against the Delta table so the change is transactional and fully logged.

```sql
UPDATE patients
SET phone_number = '+1-555-123-4567'
WHERE patient_id = 3;

SELECT * FROM patients WHERE patient_id = 3;
```

Re-querying the table immediately confirmed the update was applied correctly and consistently — no partial writes, no risk of a corrupted record.

![Locating the patients table before the update](./screenshots/image1.png)

![Verifying the phone number was updated correctly](./screenshots/image2.png)


### 1.2 — Managing prescription data: locate, then delete

Working with `prescriptions.csv`-derived data for HealthInc, I first located a specific outdated record (`prescription_id = 1001`, medication `'approach'`) and then removed it — demonstrating that deletes on Delta tables are just as transactionally safe as updates.

```sql
SELECT * FROM prescriptions
WHERE prescription_id = 1001 AND medication = 'approach';

DELETE FROM prescriptions
WHERE prescription_id = 1001 AND medication = 'approach';
```

![Locating the outdated prescription record](./screenshots/image3.png)

![Confirming the record was deleted](./screenshots/image4.png)


### 1.3 — Optimizing a high-write table with `OPTIMIZE`

The `appointments` table was receiving frequent writes, which fragments data into many small Parquet files and slows down reads (the "small file problem"). I ran Delta Lake's `OPTIMIZE` command to compact those files, then re-ran `SELECT COUNT(*)` and `DESCRIBE DETAIL` to validate the improvement in file count and query performance.

```sql
OPTIMIZE appointments;

SELECT COUNT(*) FROM appointments;
DESCRIBE DETAIL appointments;
```

**Takeaway:** the right way to verify an `OPTIMIZE` actually helped isn't to assume it did — it's to compare query execution times and file counts before and after, which is exactly what I did here.

![Row count after compaction](./screenshots/image5.png)

![DESCRIBE DETAIL showing the optimized file layout](./screenshots/image6.png)


### 1.4 — Managed vs. unmanaged tables: seeing the difference first-hand

To understand table persistence concretely (not just conceptually), I created a **managed** copy of the `patients` table and then dropped it.

```sql
CREATE TABLE patients_managed AS
SELECT * FROM patients;

DROP TABLE patients_managed;
```

**Result:** dropping a *managed* table deletes **both the metadata and the underlying data** — unlike an unmanaged table (created with a custom `LOCATION`), where the data would survive the drop. This is a distinction with real operational consequences: managed tables are simpler to maintain, but unmanaged tables are the right call when compliance requires control over where and how long the raw files persist.

![Creating the managed patients_managed table](./screenshots/image7.png)

![Confirming the managed table (and its data) is fully removed after DROP](./screenshots/image8.png)

---


## Chapter 2 — Views and Temporary Views

Views let me package a query as a reusable, named object without duplicating the underlying data. I worked with both **persistent views** (available across sessions, ideal for shared dashboards and reports) and **temp views** (scoped to a single session, ideal for scratch/staging work).


### 2.1 — Creating a persistent view over patient data

```sql
CREATE VIEW patient_view AS
SELECT * FROM patients;
```

![Creating the patient_view persistent view](./screenshots/image9.png)


### 2.2 — Aggregating with a view: patients by blood type

```sql
CREATE VIEW patient_blood_group_view AS
SELECT blood_type, COUNT(*) AS patient_count
FROM patients
GROUP BY blood_type;
```

![Creating the patient_blood_group_view aggregation view](./screenshots/image10.png)


### 2.3 — Evolving a view without breaking downstream consumers

Rather than dropping and recreating the view (which would break anything already depending on it), I used `CREATE OR REPLACE VIEW` to add a gender breakdown to the existing blood-type view.

```sql
CREATE OR REPLACE VIEW patient_blood_group_view AS
SELECT gender, blood_type, COUNT(*) AS patient_count
FROM patients
GROUP BY gender, blood_type;
```

![Updating the view with CREATE OR REPLACE VIEW](./screenshots/image11.png)


### 2.4 — Temp views: scoped to a single session

I created a **temp view** filtering patients born in or after 1980, then deliberately disconnected and reconnected to the SQL Editor to simulate a new session.

```sql
CREATE OR REPLACE TEMP VIEW patients_temp AS
SELECT * FROM patients WHERE dob >= '1980';
```

**Result:** after reconnecting, querying `patients_temp` failed — confirming temp views are session-scoped and disappear automatically, which is exactly the "auto-cleanup" behavior that makes them safe for quick, throwaway analysis.

![Creating the patients_temp temporary view](./screenshots/image12.png)

![Querying patients_temp after reconnecting — it no longer exists](./screenshots/image13.png)


### 2.5 — Persistent views don't auto-refresh, but they do persist

Finally, I created `active_patients_view` (patients born after 1980) as a **persistent** view, confirmed it existed with `SHOW VIEWS`, disconnected and reconnected again, and confirmed the view was *still* queryable — the opposite behavior of the temp view above. I then cleaned up with `DROP VIEW`.

```sql
CREATE OR REPLACE VIEW active_patients_view AS
SELECT * FROM patients WHERE dob > '1980';

SHOW VIEWS;

SELECT * FROM active_patients_view;   -- still works after reconnecting

DROP VIEW active_patients_view;
```

![Creating active_patients_view](./screenshots/image14.png)

![Confirming the view exists with SHOW VIEWS](./screenshots/image15.png)

![Querying the view successfully after reconnecting to a new session](./screenshots/image16.png)

![Dropping the view to keep the catalog clean](./screenshots/image17.png)

---


## Chapter 3 — Data Exploration, Access Control & PII

The final chapter shifted from *building* tables to *governing* them — using Data Explorer to preview and secure data, and specifically thinking about **Personally Identifiable Information (PII)**, which is central to healthcare data (patient names, insurance IDs) and subject to regulations like HIPAA and GDPR.


### 3.1 — Ingesting and previewing a new dataset through Data Explorer

I ingested `patients.csv` as a new table (`patients_new`) through the Data Ingestion UI, then used **Data Explorer** to preview its structure and grant **SELECT** (read-only) access to all account users — the appropriate access level for a dataset meant to be broadly readable but not editable.

![Previewing the newly ingested patients_new table](./screenshots/image18.png)

![Granting SELECT permission on patients_new via Data Explorer](./screenshots/image19.png)


### 3.2 — Reviewing access logs for anomalies

Monitoring who accesses sensitive tables — and when — is a core governance responsibility for a table owner. I opened the **History** tab on the `appointments` table to review recent read/write activity and check for anomalies such as unusual access times or an unexpectedly high query volume.

![Reviewing the History tab for the appointments table](./screenshots/image20.png)


### 3.3 — Verifying structure, then locking down PII

Before tightening access, I verified that the sensitive fields were correctly populated — `patient_id`, `first_name`, and `insurance_id` on `patients_new`, and `diagnosis`, `treatment_cost`, and `patient_id` on `medical_records` (checking that the two tables could be safely linked on `patient_id`). With that confirmed, I **revoked** the broad `SELECT` access I had granted earlier, restricting the sensitive patient table to only the users who genuinely need it.

![Verifying key fields on patients_new](./screenshots/image21.png)

![Verifying key fields on medical_records for safe linking](./screenshots/image22.png)

![Revoking broad SELECT access to protect PII](./screenshots/image23.png)

---


## ✅ Skills demonstrated in this module

- Performing ACID-safe `UPDATE` and `DELETE` operations on Delta Lake tables holding sensitive records.
- Using `OPTIMIZE` to solve the small-file problem and validating the improvement with `DESCRIBE DETAIL`.
- Understanding — and demonstrating first-hand — the operational difference between **managed** and **unmanaged** tables.
- Creating, evolving (`CREATE OR REPLACE`), and dropping both **persistent views** and **session-scoped temp views**.
- Using Data Explorer to ingest, preview, and govern data: granting and revoking `SELECT` permissions and reviewing access history.
- Applying PII-aware thinking to a real healthcare dataset, in line with governance frameworks like HIPAA/GDPR.
